# MR-GSM8K reference-answer issue

In the cached `Randolphzeng/Mr-GSM8K` test split (2,999 rows), 602 of 1,359 `reversed` examples (44.3%) have `ground_truth_answer == "Let's think step by step."` instead of the numerical answer. Neither `original` (1,418 rows) nor `POT` (222 rows) has this placeholder. One additional reversed answer is a sentence rather than the answer. The reversed split also has 100 rows with a missing `ground_truth_solution`; their `ground_truth_answer` values are numeric. Do not use the reversed reference-answer field without validation. The judge prompt does not include reference answers; this finding does not establish whether first-error labels are correct.

In [1]:
from collections import Counter
import re

from datasets import load_dataset

rows = load_dataset("Randolphzeng/Mr-GSM8K", split="test")
placeholder = "Let's think step by step."
for task in ("original", "POT", "reversed"):
    group = [row for row in rows if row["question_type"] == task]
    bad = [row for row in group if row["ground_truth_answer"] == placeholder]
    print(f"{task}: {len(group)} rows, {len(bad)} placeholder answers ({len(bad) / len(group):.1%}), "
          f"{sum(row['ground_truth_solution'] is None for row in group)} missing solutions")

reversed_rows = [row for row in rows if row["question_type"] == "reversed"]
non_numeric = [row for row in reversed_rows if not re.fullmatch(r"[-+]?\d+(?:,\d{3})*(?:\.\d+)?", str(row["ground_truth_answer"]))]
print("Non-numeric reversed answers:", Counter(row["ground_truth_answer"] for row in non_numeric))

original: 1418 rows, 0 placeholder answers (0.0%), 0 missing solutions
POT: 222 rows, 0 placeholder answers (0.0%), 0 missing solutions
reversed: 1359 rows, 602 placeholder answers (44.3%), 100 missing solutions
Non-numeric reversed answers: Counter({"Let's think step by step.": 602, 'Mr. Jackson wants to give each student 2 glue sticks.': 1})


In [2]:
row = next(row for row in reversed_rows if row["uuid"] == "89ecdf91-be3e-4f8c-97a1-afe93a96808d")
print("Question:", row["question"])
print("Stored ground_truth_answer:", repr(row["ground_truth_answer"]))
print("Ground-truth solution ending:", row["ground_truth_solution"].splitlines()[-1])
assert row["ground_truth_answer"] == placeholder
assert 7995 - 3075 - 2325 == 2595

Question: The Adams family is busy making cookies. So far, they've made 7995 cookies. They have x rainbow cookies, 3075 oatmeal cookies, and some chocolate chip cookies. They have made 2325 chocolate chip cookies. What is the value of unknown variable x?
Stored ground_truth_answer: "Let's think step by step."
Ground-truth solution ending: The value of x is 2595.
